# Value Research

## Stage 1C: PIT quarterly and TTM Net Income

This audit does **not** calculate Earnings Yield, Market Cap, Value scores, `X_char`, weights, or performance metrics.

## 1. Research question

Can a deterministic, point-in-time-safe sequence of economically distinct quarterly Net Income observations be reconstructed from preserved SEC EDGAR facts for AAPL, JPM, XOM, UNH, and WMT?

The future object is `NI_TTM_PIT`, not an implemented Earnings Yield formula. Economic period, filing event, and research eligibility remain separate.

## 2. Frozen economic concept

Value concept: **current economic generation relative to market valuation**. Proposed representation: Earnings Yield. Final proxy: **PENDING - requires research decision**.

Frozen PIT conventions: decision at the regular-session open on `t`; market data through completed session `t-1`; SEC facts eligible at the first eligible XNYS regular-session open strictly after `acceptanceDateTime`; scheduled early closes are eligible; amendments apply prospectively.

A versioned XNYS calendar is not yet preserved locally. This notebook shows the acceptance evidence and eligibility rule but does not invent an exact eligible-session date.

## 3. Required data and provenance

Read only immutable Stage 0B SEC `companyfacts` (`us-gaap:NetIncomeLoss`) and `submissions` evidence. Raw payloads are never modified.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 20)
FORMS = {'10-Q', '10-K', '10-Q/A', '10-K/A'}
RAW_ROOT = next(p / 'data' / 'raw' / 'stage_0b_sec' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'raw' / 'stage_0b_sec' / 'manifest.json').exists())
PROJECT_ROOT = RAW_ROOT.parents[2]
manifest = json.loads((RAW_ROOT / 'manifest.json').read_text(encoding='utf-8'))

def read_raw(entry):
    return json.loads((PROJECT_ROOT / entry['local_raw_file']).read_text(encoding='utf-8'))

print(f'Reading immutable evidence from {RAW_ROOT.relative_to(PROJECT_ROOT)}')

Reading immutable evidence from data\raw\stage_0b_sec


## 4. Filing/provenance metadata

This joins a fact's accession number to its preserved filing acceptance timestamp. XOM uses the already-audited Exxon Mobil Corporation CIK override, not the unrelated current ticker mapping.

In [2]:
resolved = {x['issuer']: x['resolved_cik'] for x in manifest if x['endpoint'] == 'identity_resolution'}
facts_payloads, filing_rows, seen = {}, [], set()

for item in manifest:
    if item['request_status'] != 'HTTP_200' or not item.get('local_raw_file'):
        continue
    if item['endpoint'] == 'companyfacts':
        payload = read_raw(item)
        expected = resolved.get(item['issuer'], str(payload.get('cik', '')).zfill(10))
        if str(payload.get('cik', '')).zfill(10) == expected:
            facts_payloads[item['issuer']] = payload
    if item['endpoint'] in {'submissions', 'submissions_history'} and item['content_sha256'] not in seen:
        seen.add(item['content_sha256'])
        payload = read_raw(item)
        if item['issuer'] in resolved:
            is_resolved_current = item['endpoint'] == 'submissions' and str(payload.get('cik', '')).zfill(10) == resolved[item['issuer']]
            is_resolved_history = item['endpoint'] == 'submissions_history' and resolved[item['issuer']] in item['local_raw_file']
            if not (is_resolved_current or is_resolved_history):
                continue
        inventory = payload.get('filings', {}).get('recent', payload)
        for values in zip(*inventory.values()):
            row = dict(zip(inventory, values))
            if row.get('form') in FORMS:
                filing_rows.append({'issuer': item['issuer'], **row})

filings = (pd.DataFrame(filing_rows).drop_duplicates(['issuer', 'accessionNumber'])
           [['issuer', 'accessionNumber', 'form', 'reportDate', 'filingDate', 'acceptanceDateTime']])
filings['acceptanceDateTime'] = pd.to_datetime(filings['acceptanceDateTime'], utc=True, errors='coerce')
filings.groupby('issuer').agg(filings=('accessionNumber', 'size'), amendments=('form', lambda x: x.str.endswith('/A').sum()), missing_acceptance=('acceptanceDateTime', lambda x: x.isna().sum()))

,filings,amendments,missing_acceptance
issuer,,,
AAPL,132,4,0
JPM,137,12,0
UNH,132,8,0
WMT,132,5,0
XOM,132,3,0


## 5. Raw Net Income facts and duration analysis

`fp` is displayed but is not proof of a standalone quarter. Duration comes directly from `end - start`; labels below are descriptive audit labels, not a reconstruction rule.

In [3]:
rows = []
for issuer, payload in facts_payloads.items():
    concept = payload.get('facts', {}).get('us-gaap', {}).get('NetIncomeLoss', {})
    for unit, observations in concept.get('units', {}).items():
        for x in observations:
            if x.get('form') in FORMS:
                rows.append({'issuer': issuer, 'cik': str(payload['cik']).zfill(10), 'taxonomy': 'us-gaap', 'tag': 'NetIncomeLoss', 'unit': unit, 'start': x.get('start'), 'end': x.get('end'), 'value': x.get('val'), 'fy': x.get('fy'), 'fp': x.get('fp'), 'form': x.get('form'), 'filed': x.get('filed'), 'accn': x.get('accn'), 'frame': x.get('frame')})

net_income = pd.DataFrame(rows).merge(filings.rename(columns={'accessionNumber': 'accn', 'form': 'filing_form'}), on=['issuer', 'accn'], how='left', validate='many_to_one')
net_income['start'] = pd.to_datetime(net_income['start'], errors='coerce')
net_income['end'] = pd.to_datetime(net_income['end'], errors='coerce')
net_income['duration_days'] = (net_income['end'] - net_income['start']).dt.days
net_income['duration_interpretation'] = pd.cut(net_income['duration_days'], [-1, 110, 200, 300, float('inf')], labels=['quarter-duration candidate', 'H1/YTD-duration candidate', '9M/YTD-duration candidate', 'annual-duration candidate'])
net_income['pit_eligibility'] = 'First eligible XNYS open strictly after acceptanceDateTime; exact date PENDING versioned XNYS calendar'

show = ['issuer', 'cik', 'taxonomy', 'tag', 'start', 'end', 'duration_days', 'value', 'unit', 'fy', 'fp', 'form', 'filed', 'accn', 'frame', 'acceptanceDateTime', 'pit_eligibility']
net_income[show].head(10)

,issuer,cik,taxonomy,tag,start,end,duration_days,value,unit,fy,fp,form,filed,accn,frame,acceptanceDateTime,pit_eligibility
0,AAPL,0000320193,us-gaap,NetIncomeLoss,2006-10-01,2007-09-29,363,3496000000,USD,2009,FY,10-K,2009-10-27,0001193125-09-214859,None,2009-10-27 20:18:29+00:00,First eligible XNYS open strictly after accept...
1,AAPL,0000320193,us-gaap,NetIncomeLoss,2006-10-01,2007-09-29,363,3495000000,USD,2009,FY,10-K/A,2010-01-25,0001193125-10-012091,CY2007,2010-01-25 21:25:58+00:00,First eligible XNYS open strictly after accept...
2,AAPL,0000320193,us-gaap,NetIncomeLoss,2007-09-30,2008-06-28,272,3698000000,USD,2009,Q3,10-Q,2009-07-22,0001193125-09-153165,None,2009-07-22 20:41:33+00:00,First eligible XNYS open strictly after accept...
3,AAPL,0000320193,us-gaap,NetIncomeLoss,2008-03-30,2008-06-28,90,1072000000,USD,2009,Q3,10-Q,2009-07-22,0001193125-09-153165,CY2008Q2,2009-07-22 20:41:33+00:00,First eligible XNYS open strictly after accept...
4,AAPL,0000320193,us-gaap,NetIncomeLoss,2007-09-30,2008-09-27,363,4834000000,USD,2009,FY,10-K,2009-10-27,0001193125-09-214859,None,2009-10-27 20:18:29+00:00,First eligible XNYS open strictly after accept...
5,AAPL,0000320193,us-gaap,NetIncomeLoss,2007-09-30,2008-09-27,363,6119000000,USD,2009,FY,10-K/A,2010-01-25,0001193125-10-012091,None,2010-01-25 21:25:58+00:00,First eligible XNYS open strictly after accept...
6,AAPL,0000320193,us-gaap,NetIncomeLoss,2007-09-30,2008-09-27,363,6119000000,USD,2010,FY,10-K,2010-10-27,0001193125-10-238044,CY2008,2010-10-27 20:36:21+00:00,First eligible XNYS open strictly after accept...
7,AAPL,0000320193,us-gaap,NetIncomeLoss,2008-09-28,2008-12-27,90,2255000000,USD,2010,Q1,10-Q,2010-01-25,0001193125-10-012085,None,2010-01-25 21:23:57+00:00,First eligible XNYS open strictly after accept...
8,AAPL,0000320193,us-gaap,NetIncomeLoss,2008-09-28,2008-12-27,90,2255000000,USD,2010,FY,10-K,2010-10-27,0001193125-10-238044,CY2008Q4,2010-10-27 20:36:21+00:00,First eligible XNYS open strictly after accept...
9,AAPL,0000320193,us-gaap,NetIncomeLoss,2008-09-28,2009-03-28,181,3875000000,USD,2010,Q2,10-Q,2010-04-21,0001193125-10-088957,None,2010-04-21 20:39:23+00:00,First eligible XNYS open strictly after accept...


In [4]:
duration_summary = (net_income.groupby(['issuer', 'form', 'fp', 'duration_interpretation'], observed=False).size().rename('facts').reset_index())
duration_summary.query('facts > 0').sort_values(['issuer', 'form', 'fp', 'duration_interpretation'])

,issuer,form,fp,duration_interpretation,facts
0,AAPL,10-K,FY,quarter-duration candidate,88
3,AAPL,10-K,FY,annual-duration candidate,51
19,AAPL,10-K/A,FY,annual-duration candidate,3
36,AAPL,10-Q,Q1,quarter-duration candidate,34
40,AAPL,10-Q,Q2,quarter-duration candidate,34
41,AAPL,10-Q,Q2,H1/YTD-duration candidate,34
44,AAPL,10-Q,Q3,quarter-duration candidate,36
46,AAPL,10-Q,Q3,9M/YTD-duration candidate,36
67,JPM,10-K,FY,annual-duration candidate,51
100,JPM,10-Q,Q1,quarter-duration candidate,34


## 6. Representative examples

Small samples demonstrate a quarter-duration candidate and a cumulative/YTD-duration candidate. They are not values used in a Value characteristic.

In [5]:
display_columns = ['issuer', 'start', 'end', 'duration_days', 'value', 'fy', 'fp', 'form', 'filed', 'accn', 'frame', 'acceptanceDateTime', 'pit_eligibility']
clean = net_income.loc[net_income['duration_interpretation'].eq('quarter-duration candidate') & net_income['form'].eq('10-Q')]
clean.sort_values(['issuer', 'end', 'acceptanceDateTime']).head(8)[display_columns]

,issuer,start,end,duration_days,value,fy,fp,form,filed,accn,frame,acceptanceDateTime,pit_eligibility
3,AAPL,2008-03-30,2008-06-28,90,1072000000,2009,Q3,10-Q,2009-07-22,0001193125-09-153165,CY2008Q2,2009-07-22 20:41:33+00:00,First eligible XNYS open strictly after accept...
7,AAPL,2008-09-28,2008-12-27,90,2255000000,2010,Q1,10-Q,2010-01-25,0001193125-10-012085,None,2010-01-25 21:23:57+00:00,First eligible XNYS open strictly after accept...
10,AAPL,2008-12-28,2009-03-28,90,1620000000,2010,Q2,10-Q,2010-04-21,0001193125-10-088957,None,2010-04-21 20:39:23+00:00,First eligible XNYS open strictly after accept...
14,AAPL,2009-03-29,2009-06-27,90,1229000000,2009,Q3,10-Q,2009-07-22,0001193125-09-153165,None,2009-07-22 20:41:33+00:00,First eligible XNYS open strictly after accept...
15,AAPL,2009-03-29,2009-06-27,90,1828000000,2010,Q3,10-Q,2010-07-21,0001193125-10-162840,None,2010-07-21 20:37:45+00:00,First eligible XNYS open strictly after accept...
22,AAPL,2009-09-27,2009-12-26,90,3378000000,2010,Q1,10-Q,2010-01-25,0001193125-10-012085,None,2010-01-25 21:23:57+00:00,First eligible XNYS open strictly after accept...
24,AAPL,2009-09-27,2009-12-26,90,3378000000,2011,Q1,10-Q,2011-01-19,0001193125-11-010144,None,2011-01-19 21:33:29+00:00,First eligible XNYS open strictly after accept...
28,AAPL,2009-12-27,2010-03-27,90,3074000000,2010,Q2,10-Q,2010-04-21,0001193125-10-088957,None,2010-04-21 20:39:23+00:00,First eligible XNYS open strictly after accept...


In [6]:
cumulative = net_income.loc[net_income['duration_interpretation'].isin(['H1/YTD-duration candidate', '9M/YTD-duration candidate'])]
cumulative.sort_values(['issuer', 'end', 'duration_days']).head(8)[display_columns]

,issuer,start,end,duration_days,value,fy,fp,form,filed,accn,frame,acceptanceDateTime,pit_eligibility
2,AAPL,2007-09-30,2008-06-28,272,3698000000,2009,Q3,10-Q,2009-07-22,0001193125-09-153165,None,2009-07-22 20:41:33+00:00,First eligible XNYS open strictly after accept...
9,AAPL,2008-09-28,2009-03-28,181,3875000000,2010,Q2,10-Q,2010-04-21,0001193125-10-088957,None,2010-04-21 20:39:23+00:00,First eligible XNYS open strictly after accept...
12,AAPL,2008-09-28,2009-06-27,272,4039000000,2009,Q3,10-Q,2009-07-22,0001193125-09-153165,None,2009-07-22 20:41:33+00:00,First eligible XNYS open strictly after accept...
13,AAPL,2008-09-28,2009-06-27,272,5703000000,2010,Q3,10-Q,2010-07-21,0001193125-10-162840,None,2010-07-21 20:37:45+00:00,First eligible XNYS open strictly after accept...
26,AAPL,2009-09-27,2010-03-27,181,6452000000,2010,Q2,10-Q,2010-04-21,0001193125-10-088957,None,2010-04-21 20:39:23+00:00,First eligible XNYS open strictly after accept...
27,AAPL,2009-09-27,2010-03-27,181,6452000000,2011,Q2,10-Q,2011-04-21,0001193125-11-104388,None,2011-04-21 10:13:41+00:00,First eligible XNYS open strictly after accept...
32,AAPL,2009-09-27,2010-06-26,272,9705000000,2010,Q3,10-Q,2010-07-21,0001193125-10-162840,None,2010-07-21 20:37:45+00:00,First eligible XNYS open strictly after accept...
33,AAPL,2009-09-27,2010-06-26,272,9705000000,2011,Q3,10-Q,2011-07-20,0001193125-11-192493,None,2011-07-20 20:32:13+00:00,First eligible XNYS open strictly after accept...


## 7. Duplicate economic periods and amendments

A duplicate flag means the same issuer/tag/unit/start/end occurs in more than one record. It is an inspection flag, not a deduplication rule. Amendments remain distinct filing events and are only eligible prospectively.

In [7]:
economic_key = ['issuer', 'tag', 'unit', 'start', 'end']
net_income['duplicate_economic_period'] = net_income.duplicated(economic_key, keep=False)
duplicates = net_income.loc[net_income['duplicate_economic_period']].sort_values(economic_key + ['acceptanceDateTime'])
amendments = net_income.loc[net_income['form'].str.endswith('/A')].sort_values(['issuer', 'acceptanceDateTime'])

print('Duplicate economic-period examples')
duplicates[display_columns].head(12)

Duplicate economic-period examples


,issuer,start,end,duration_days,value,fy,fp,form,filed,accn,frame,acceptanceDateTime,pit_eligibility
0,AAPL,2006-10-01,2007-09-29,363,3496000000,2009,FY,10-K,2009-10-27,0001193125-09-214859,None,2009-10-27 20:18:29+00:00,First eligible XNYS open strictly after accept...
1,AAPL,2006-10-01,2007-09-29,363,3495000000,2009,FY,10-K/A,2010-01-25,0001193125-10-012091,CY2007,2010-01-25 21:25:58+00:00,First eligible XNYS open strictly after accept...
4,AAPL,2007-09-30,2008-09-27,363,4834000000,2009,FY,10-K,2009-10-27,0001193125-09-214859,None,2009-10-27 20:18:29+00:00,First eligible XNYS open strictly after accept...
5,AAPL,2007-09-30,2008-09-27,363,6119000000,2009,FY,10-K/A,2010-01-25,0001193125-10-012091,None,2010-01-25 21:25:58+00:00,First eligible XNYS open strictly after accept...
6,AAPL,2007-09-30,2008-09-27,363,6119000000,2010,FY,10-K,2010-10-27,0001193125-10-238044,CY2008,2010-10-27 20:36:21+00:00,First eligible XNYS open strictly after accept...
7,AAPL,2008-09-28,2008-12-27,90,2255000000,2010,Q1,10-Q,2010-01-25,0001193125-10-012085,None,2010-01-25 21:23:57+00:00,First eligible XNYS open strictly after accept...
8,AAPL,2008-09-28,2008-12-27,90,2255000000,2010,FY,10-K,2010-10-27,0001193125-10-238044,CY2008Q4,2010-10-27 20:36:21+00:00,First eligible XNYS open strictly after accept...
12,AAPL,2008-09-28,2009-06-27,272,4039000000,2009,Q3,10-Q,2009-07-22,0001193125-09-153165,None,2009-07-22 20:41:33+00:00,First eligible XNYS open strictly after accept...
13,AAPL,2008-09-28,2009-06-27,272,5703000000,2010,Q3,10-Q,2010-07-21,0001193125-10-162840,None,2010-07-21 20:37:45+00:00,First eligible XNYS open strictly after accept...
17,AAPL,2008-09-28,2009-09-26,363,5704000000,2009,FY,10-K,2009-10-27,0001193125-09-214859,None,2009-10-27 20:18:29+00:00,First eligible XNYS open strictly after accept...


In [8]:
print('Amendment examples')
amendments[display_columns].head(12)

Amendment examples


,issuer,start,end,duration_days,value,fy,fp,form,filed,accn,frame,acceptanceDateTime,pit_eligibility
1,AAPL,2006-10-01,2007-09-29,363,3495000000,2009,FY,10-K/A,2010-01-25,0001193125-10-012091,CY2007,2010-01-25 21:25:58+00:00,First eligible XNYS open strictly after accept...
5,AAPL,2007-09-30,2008-09-27,363,6119000000,2009,FY,10-K/A,2010-01-25,0001193125-10-012091,None,2010-01-25 21:25:58+00:00,First eligible XNYS open strictly after accept...
18,AAPL,2008-09-28,2009-09-26,363,8235000000,2009,FY,10-K/A,2010-01-25,0001193125-10-012091,None,2010-01-25 21:25:58+00:00,First eligible XNYS open strictly after accept...
350,JPM,2011-01-01,2011-03-31,89,5555000000,2012,Q1,10-Q/A,2012-08-09,0000019617-12-000262,CY2011Q1,2012-08-09 11:44:21+00:00,First eligible XNYS open strictly after accept...
363,JPM,2012-01-01,2012-03-31,90,4924000000,2012,Q1,10-Q/A,2012-08-09,0000019617-12-000262,None,2012-08-09 11:44:21+00:00,First eligible XNYS open strictly after accept...
869,WMT,2012-02-01,2012-07-31,181,7758000000,2013,Q2,10-Q/A,2013-10-21,0000104169-13-000039,None,2013-10-21 21:25:31+00:00,First eligible XNYS open strictly after accept...
873,WMT,2012-05-01,2012-07-31,91,4016000000,2013,Q2,10-Q/A,2013-10-21,0000104169-13-000039,None,2013-10-21 21:25:31+00:00,First eligible XNYS open strictly after accept...
891,WMT,2013-02-01,2013-07-31,180,7853000000,2013,Q2,10-Q/A,2013-10-21,0000104169-13-000039,None,2013-10-21 21:25:31+00:00,First eligible XNYS open strictly after accept...
894,WMT,2013-05-01,2013-07-31,91,4069000000,2013,Q2,10-Q/A,2013-10-21,0000104169-13-000039,None,2013-10-21 21:25:31+00:00,First eligible XNYS open strictly after accept...
1114,XOM,2008-01-01,2008-12-31,365,45220000000,2010,FY,10-K/A,2011-02-28,0001193125-11-050134,CY2008,2011-02-28 22:01:32+00:00,First eligible XNYS open strictly after accept...


## 8. Timeline-style provenance example

This small AAPL timeline separates economic period, filing, acceptance, and the PIT eligibility rule.

In [9]:
timeline = (net_income.loc[net_income['issuer'].eq('AAPL')].sort_values(['acceptanceDateTime', 'end']).tail(12).assign(economic_period=lambda x: x['start'].dt.date.astype(str) + ' to ' + x['end'].dt.date.astype(str), filing=lambda x: x['form'] + ' / ' + x['accn'])[['economic_period', 'filing', 'acceptanceDateTime', 'pit_eligibility', 'duration_interpretation', 'value']])
timeline

,economic_period,filing,acceptanceDateTime,pit_eligibility,duration_interpretation,value
299,2023-10-01 to 2024-09-28,10-K / 0000320193-25-000079,2025-10-31 10:01:26+00:00,First eligible XNYS open strictly after accept...,annual-duration candidate,93736000000
310,2024-09-29 to 2025-09-27,10-K / 0000320193-25-000079,2025-10-31 10:01:26+00:00,First eligible XNYS open strictly after accept...,annual-duration candidate,112010000000
301,2024-09-29 to 2024-12-28,10-Q / 0000320193-26-000006,2026-01-30 11:01:32+00:00,First eligible XNYS open strictly after accept...,quarter-duration candidate,36330000000
311,2025-09-28 to 2025-12-27,10-Q / 0000320193-26-000006,2026-01-30 11:01:32+00:00,First eligible XNYS open strictly after accept...,quarter-duration candidate,42097000000
303,2024-09-29 to 2025-03-29,10-Q / 0000320193-26-000013,2026-05-01 10:01:00+00:00,First eligible XNYS open strictly after accept...,H1/YTD-duration candidate,61110000000
305,2024-12-29 to 2025-03-29,10-Q / 0000320193-26-000013,2026-05-01 10:01:00+00:00,First eligible XNYS open strictly after accept...,quarter-duration candidate,24780000000
312,2025-09-28 to 2026-03-28,10-Q / 0000320193-26-000013,2026-05-01 10:01:00+00:00,First eligible XNYS open strictly after accept...,H1/YTD-duration candidate,71675000000
313,2025-12-28 to 2026-03-28,10-Q / 0000320193-26-000013,2026-05-01 10:01:00+00:00,First eligible XNYS open strictly after accept...,quarter-duration candidate,29578000000
307,2024-09-29 to 2025-06-28,10-Q / 0000320193-26-000020,2026-07-31 10:01:02+00:00,First eligible XNYS open strictly after accept...,9M/YTD-duration candidate,84544000000
309,2025-03-30 to 2025-06-28,10-Q / 0000320193-26-000020,2026-07-31 10:01:02+00:00,First eligible XNYS open strictly after accept...,quarter-duration candidate,23434000000


## 9. AAPL traced issuer: raw fact to economic-quarter interpretation

The next cells select a small AAPL fiscal-year example transparently. The selected vintage is the earliest accession for each period that was accepted no later than the annual filing used for the Q4 candidate. This is an **audit reconciliation**, not a frozen reconstruction implementation.

`raw SEC facts -> selected accession/vintage -> Stage 1B PIT eligibility rule -> economic-quarter interpretation`

In [10]:
aapl = net_income.loc[net_income['issuer'].eq('AAPL')].copy()
annual = (aapl.loc[aapl['form'].eq('10-K') & aapl['duration_interpretation'].eq('annual-duration candidate')]
          .sort_values(['end', 'acceptanceDateTime'])
          .iloc[-1])
fy_start, fy_end, annual_acceptance = annual['start'], annual['end'], annual['acceptanceDateTime']

# Only facts accepted by the annual filing's vintage may support this audit's Q4 candidate.
within_fy = aapl.loc[(aapl['start'] >= fy_start) & (aapl['end'] <= fy_end) & (aapl['acceptanceDateTime'] <= annual_acceptance)]
direct_pool = within_fy.loc[within_fy['form'].eq('10-Q') & within_fy['duration_interpretation'].eq('quarter-duration candidate')]
direct_by_end = (direct_pool.sort_values('acceptanceDateTime')
                 .drop_duplicates(['start', 'end'], keep='first')
                 .sort_values('end'))
q1_direct, q2_direct, q3_direct = [direct_by_end.iloc[i] for i in range(3)]
h1_ytd = (within_fy.loc[within_fy['duration_interpretation'].eq('H1/YTD-duration candidate')]
          .sort_values('acceptanceDateTime')
          .drop_duplicates(['start', 'end'], keep='first')
          .iloc[-1])

trace = pd.DataFrame([
    {'case': 'direct quarter', 'raw period': f"{q1_direct['start'].date()} to {q1_direct['end'].date()}", 'accession / vintage': q1_direct['accn'], 'acceptance': q1_direct['acceptanceDateTime'], 'raw value': q1_direct['value'], 'interpretation': 'Q1 direct-quarter candidate'},
    {'case': 'YTD operand', 'raw period': f"{h1_ytd['start'].date()} to {h1_ytd['end'].date()}", 'accession / vintage': h1_ytd['accn'], 'acceptance': h1_ytd['acceptanceDateTime'], 'raw value': h1_ytd['value'], 'interpretation': 'H1/YTD; not a second standalone quarter'},
    {'case': 'direct comparison', 'raw period': f"{q2_direct['start'].date()} to {q2_direct['end'].date()}", 'accession / vintage': q2_direct['accn'], 'acceptance': q2_direct['acceptanceDateTime'], 'raw value': q2_direct['value'], 'interpretation': 'Q2 direct-quarter candidate'},
    {'case': 'Q4 operand', 'raw period': f"{annual['start'].date()} to {annual['end'].date()}", 'accession / vintage': annual['accn'], 'acceptance': annual['acceptanceDateTime'], 'raw value': annual['value'], 'interpretation': 'FY annual fact; candidate Q4 source'},
])
trace['PIT eligible'] = 'First eligible XNYS open strictly after acceptance; exact date pending versioned calendar'
print(f"AAPL fiscal period traced: {fy_start.date()} to {fy_end.date()}")
trace

AAPL fiscal period traced: 2024-09-29 to 2025-09-27


,case,raw period,accession / vintage,acceptance,raw value,interpretation,PIT eligible
0,direct quarter,2024-09-29 to 2024-12-28,0000320193-25-000008,2025-01-31 11:01:27+00:00,36330000000,Q1 direct-quarter candidate,First eligible XNYS open strictly after accept...
1,YTD operand,2024-09-29 to 2025-03-29,0000320193-25-000057,2025-05-02 10:00:46+00:00,61110000000,H1/YTD; not a second standalone quarter,First eligible XNYS open strictly after accept...
2,direct comparison,2024-12-29 to 2025-03-29,0000320193-25-000057,2025-05-02 10:00:46+00:00,24780000000,Q2 direct-quarter candidate,First eligible XNYS open strictly after accept...
3,Q4 operand,2024-09-29 to 2025-09-27,0000320193-25-000079,2025-10-31 10:01:26+00:00,112010000000,FY annual fact; candidate Q4 source,First eligible XNYS open strictly after accept...


In [11]:
q2_from_ytd = h1_ytd['value'] - q1_direct['value']
q4_candidate = annual['value'] - q1_direct['value'] - q2_direct['value'] - q3_direct['value']

reconciliation = pd.DataFrame([
    {'interpretation': 'Q1 direct', 'raw expression': 'Q1 quarter-duration fact', 'audit value': q1_direct['value']},
    {'interpretation': 'Q2 via YTD subtraction', 'raw expression': 'H1/YTD fact - Q1 direct fact', 'audit value': q2_from_ytd},
    {'interpretation': 'Q2 direct comparison', 'raw expression': 'Q2 quarter-duration fact', 'audit value': q2_direct['value']},
    {'interpretation': 'Q4 candidate', 'raw expression': 'FY annual fact - Q1 direct - Q2 direct - Q3 direct', 'audit value': q4_candidate},
])
reconciliation

,interpretation,raw expression,audit value
0,Q1 direct,Q1 quarter-duration fact,36330000000
1,Q2 via YTD subtraction,H1/YTD fact - Q1 direct fact,24780000000
2,Q2 direct comparison,Q2 quarter-duration fact,24780000000
3,Q4 candidate,FY annual fact - Q1 direct - Q2 direct - Q3 di...,27466000000


In [12]:
aapl_amendment = aapl.loc[aapl['form'].str.endswith('/A')].sort_values('acceptanceDateTime').iloc[0]
same_period = (aapl.loc[(aapl['start'].eq(aapl_amendment['start'])) & (aapl['end'].eq(aapl_amendment['end']))]
               .sort_values('acceptanceDateTime'))
amendment_trace = same_period[['start', 'end', 'value', 'form', 'filed', 'accn', 'acceptanceDateTime', 'frame']].copy()
amendment_trace['economic interpretation'] = 'Same reported period; later accession is prospectively eligible only'
amendment_trace

,start,end,value,form,filed,accn,acceptanceDateTime,frame,economic interpretation
0,2006-10-01,2007-09-29,3496000000,10-K,2009-10-27,0001193125-09-214859,2009-10-27 20:18:29+00:00,None,Same reported period; later accession is prosp...
1,2006-10-01,2007-09-29,3495000000,10-K/A,2010-01-25,0001193125-10-012091,2010-01-25 21:25:58+00:00,CY2007,Same reported period; later accession is prosp...


## 10. Reconstruction feasibility

Do **not** sort and sum four fact rows. A quarter-duration fact is only a direct-quarter candidate after exact period/context and filing-vintage selection. H1/YTD and 9M/YTD observations are cumulative by duration; a future implementation could use subtraction only after a frozen rule and PIT checks on both operands. Q4 may require annual Net Income less the first three economically distinct quarters, each eligible by the annual filing's own PIT date. Later annual filings and amendments may not backfill earlier information sets.

In [13]:
inventory = (net_income.groupby(['issuer', 'duration_interpretation'], observed=False).agg(facts=('accn', 'size'), accession_events=('accn', 'nunique'), amendments=('form', lambda x: x.str.endswith('/A').sum()), duplicate_rows=('duplicate_economic_period', 'sum')).reset_index())
inventory.query('facts > 0').sort_values(['issuer', 'duration_interpretation'])

,issuer,duration_interpretation,facts,accession_events,amendments,duplicate_rows
0,AAPL,quarter-duration candidate,192,63,0,186
1,AAPL,H1/YTD-duration candidate,34,17,0,32
2,AAPL,9M/YTD-duration candidate,36,18,0,34
3,AAPL,annual-duration candidate,54,18,3,53
4,JPM,quarter-duration candidate,106,53,2,100
5,JPM,H1/YTD-duration candidate,36,18,0,34
6,JPM,9M/YTD-duration candidate,34,17,0,32
7,JPM,annual-duration candidate,51,17,0,49
8,UNH,quarter-duration candidate,134,55,0,125
9,UNH,H1/YTD-duration candidate,36,18,0,34


## 11. Falsification test across JPM, XOM, UNH, and WMT

This is an attempt to break the candidate reconstruction methodology, not a search for confirming examples. For each issuer, the notebook takes a recent complete fiscal year and tests direct facts, cumulative subtraction, the Q4 annual-minus-9M candidate, period structure, units, concept, and filing vintage. Numerical equality is necessary but not sufficient.

The selection procedure is deliberately visible below: take the latest annual 10-K period, retain only facts accepted no later than that annual filing, and select the earliest accepted vintage for an exact period. These are **candidate audit rules**, not frozen implementation rules.

In [14]:
TEST_ISSUERS = ['JPM', 'XOM', 'UNH', 'WMT']

def earliest_vintage(frame):
    return frame.sort_values(['acceptanceDateTime', 'accn']).iloc[0] if not frame.empty else None

validation_rows, quarter_rows = [], []
for issuer in TEST_ISSUERS:
    issuer_facts = net_income.loc[net_income['issuer'].eq(issuer)].copy()
    annual_pool = issuer_facts.loc[issuer_facts['form'].eq('10-K') & issuer_facts['duration_interpretation'].eq('annual-duration candidate')]
    annual_end = annual_pool['end'].max()
    annual = earliest_vintage(annual_pool.loc[annual_pool['end'].eq(annual_end)])
    fiscal_start, fiscal_end, annual_acceptance = annual['start'], annual['end'], annual['acceptanceDateTime']

    # This prevents later comparative filings from entering the annual-vintage audit.
    fiscal_facts = issuer_facts.loc[(issuer_facts['start'] >= fiscal_start) & (issuer_facts['end'] <= fiscal_end) & (issuer_facts['acceptanceDateTime'] <= annual_acceptance)]
    direct = fiscal_facts.loc[fiscal_facts['form'].eq('10-Q') & fiscal_facts['duration_interpretation'].eq('quarter-duration candidate')]
    q1 = earliest_vintage(direct.loc[direct['start'].eq(fiscal_start)])
    h1 = earliest_vintage(fiscal_facts.loc[fiscal_facts['start'].eq(fiscal_start) & fiscal_facts['duration_interpretation'].eq('H1/YTD-duration candidate')])
    y9 = earliest_vintage(fiscal_facts.loc[fiscal_facts['start'].eq(fiscal_start) & fiscal_facts['duration_interpretation'].eq('9M/YTD-duration candidate')])
    q2 = earliest_vintage(direct.loc[direct['end'].eq(h1['end'])]) if h1 is not None else None
    q3 = earliest_vintage(direct.loc[direct['end'].eq(y9['end'])]) if y9 is not None else None

    basic = all(item is not None for item in [q1, h1, y9, annual])
    compatible = basic and all(item['issuer'] == issuer and item['tag'] == 'NetIncomeLoss' and item['unit'] == annual['unit'] for item in [q1, h1, y9, annual])
    structure = compatible and q1['start'] == fiscal_start and h1['start'] == fiscal_start and y9['start'] == fiscal_start and h1['end'] < y9['end'] < fiscal_end
    vintage = basic and q1['acceptanceDateTime'] <= h1['acceptanceDateTime'] <= y9['acceptanceDateTime'] <= annual_acceptance
    q2_reconstructed = h1['value'] - q1['value'] if basic and structure and vintage else None
    q3_reconstructed = y9['value'] - h1['value'] if basic and structure and vintage else None
    q4_reconstructed = annual['value'] - y9['value'] if basic and structure and vintage else None
    q2_difference = q2['value'] - q2_reconstructed if q2 is not None and q2_reconstructed is not None else None
    q3_difference = q3['value'] - q3_reconstructed if q3 is not None and q3_reconstructed is not None else None

    duplicate_count = fiscal_facts.duplicated(['issuer', 'tag', 'unit', 'start', 'end'], keep=False).sum()
    amendment_count = fiscal_facts['form'].str.endswith('/A').sum()
    issues = []
    if duplicate_count: issues.append(f'{duplicate_count} duplicate/comparative rows')
    if amendment_count: issues.append(f'{amendment_count} amendment rows')
    if not structure: issues.append('period structure not proven compatible')
    if not vintage: issues.append('acceptance chronology not proven')
    if q2 is None or q3 is None: issues.append('missing direct check')
    if q2_difference not in (None, 0) or q3_difference not in (None, 0): issues.append('direct/reconstructed mismatch')
    if not basic or not compatible or not structure or not vintage or q2_difference not in (None, 0) or q3_difference not in (None, 0):
        result = 'FAIL'
    elif issues:
        result = 'PASS WITH CONDITION'
    else:
        result = 'PASS'

    validation_rows.append({'issuer': issuer, 'fiscal_year': annual['fy'], 'fiscal_period': f"{fiscal_start.date()} to {fiscal_end.date()}", 'Q1_direct_available': 'PASS' if q1 is not None else 'FAIL', 'Q2_reconstructable': 'PASS' if q2_reconstructed is not None else 'FAIL', 'Q2_direct_check': q2_difference, 'Q3_reconstructable': 'PASS' if q3_reconstructed is not None else 'FAIL', 'Q3_direct_check': q3_difference, 'Q4_reconstructable': 'PASS' if q4_reconstructed is not None else 'FAIL', 'duplicate_or_vintage_issue': '; '.join(issues) or 'none observed', 'result': result})

    def source_text(items):
        return ' | '.join(f"{x['form']} {x['accn']}" for x in items if x is not None)
    def period_text(items):
        return ' | '.join(f"{x['start'].date()} to {x['end'].date()} ({x['duration_days']}d)" for x in items if x is not None)
    def acceptance_text(items):
        return ' | '.join(str(x['acceptanceDateTime']) for x in items if x is not None)
    for quarter, sources, direct_fact, reconstructed, difference, interpretation in [
        ('Q1', [q1], q1, q1['value'] if q1 is not None else None, None, 'direct quarter candidate'),
        ('Q2', [h1, q1], q2, q2_reconstructed, q2_difference, 'H1/YTD - Q1'),
        ('Q3', [y9, h1], q3, q3_reconstructed, q3_difference, '9M/YTD - H1/YTD'),
        ('Q4', [annual, y9], None, q4_reconstructed, None, 'FY - 9M/YTD candidate'),
    ]:
        quarter_rows.append({'issuer': issuer, 'fiscal_year': annual['fy'], 'quarter': quarter, 'candidate interpretation': interpretation, 'source accession(s)': source_text(sources), 'source form(s)': ' | '.join(x['form'] for x in sources if x is not None), 'source start/end/duration': period_text(sources), 'source value(s)': ' | '.join(str(x['value']) for x in sources if x is not None), 'acceptance timestamp(s)': acceptance_text(sources), 'direct accession/form': None if direct_fact is None else f"{direct_fact['accn']} / {direct_fact['form']}", 'direct period/duration': None if direct_fact is None else f"{direct_fact['start'].date()} to {direct_fact['end'].date()} ({direct_fact['duration_days']}d)", 'direct acceptance': None if direct_fact is None else direct_fact['acceptanceDateTime'], 'direct-quarter value': direct_fact['value'] if direct_fact is not None else None, 'reconstructed value': reconstructed, 'direct minus reconstructed': difference, 'contexts comparable': compatible and structure and vintage})

validation = pd.DataFrame(validation_rows)
quarter_validation = pd.DataFrame(quarter_rows)
validation

,issuer,fiscal_year,fiscal_period,Q1_direct_available,Q2_reconstructable,Q2_direct_check,Q3_reconstructable,Q3_direct_check,Q4_reconstructable,duplicate_or_vintage_issue,result
0,JPM,2025,2025-01-01 to 2025-12-31,PASS,PASS,0,PASS,0,PASS,none observed,PASS
1,XOM,2025,2025-01-01 to 2025-12-31,PASS,PASS,0,PASS,0,PASS,none observed,PASS
2,UNH,2025,2025-01-01 to 2025-12-31,PASS,PASS,0,PASS,0,PASS,none observed,PASS
3,WMT,2026,2025-02-01 to 2026-01-31,PASS,PASS,0,PASS,0,PASS,none observed,PASS


### Quarter-level source and reconstruction evidence

A `0` direct-minus-reconstructed difference is only a reconciliation result. The final column also requires compatible issuer, concept, USD unit, fiscal start/end structure, and acceptance chronology. The exact XNYS PIT session remains pending the versioned calendar.

In [15]:
quarter_validation.sort_values(['issuer', 'quarter'])

,issuer,fiscal_year,quarter,candidate interpretation,source accession(s),source form(s),source start/end/duration,source value(s),acceptance timestamp(s),direct accession/form,direct period/duration,direct acceptance,direct-quarter value,reconstructed value,direct minus reconstructed,contexts comparable
0,JPM,2025,Q1,direct quarter candidate,10-Q 0000019617-25-000421,10-Q,2025-01-01 to 2025-03-31 (89d),14643000000,2025-05-01 20:16:56+00:00,0000019617-25-000421 / 10-Q,2025-01-01 to 2025-03-31 (89d),2025-05-01 20:16:56+00:00,1.464300e+10,14643000000,NaN,True
1,JPM,2025,Q2,H1/YTD - Q1,10-Q 0000019617-25-000615 | 10-Q 0000019617-25...,10-Q | 10-Q,2025-01-01 to 2025-06-30 (180d) | 2025-01-01 t...,29630000000 | 14643000000,2025-08-05 20:18:22+00:00 | 2025-05-01 20:16:5...,0000019617-25-000615 / 10-Q,2025-04-01 to 2025-06-30 (90d),2025-08-05 20:18:22+00:00,1.498700e+10,14987000000,0.0,True
2,JPM,2025,Q3,9M/YTD - H1/YTD,10-Q 0001628280-25-048859 | 10-Q 0000019617-25...,10-Q | 10-Q,2025-01-01 to 2025-09-30 (272d) | 2025-01-01 t...,44023000000 | 29630000000,2025-11-04 21:18:08+00:00 | 2025-08-05 20:18:2...,0001628280-25-048859 / 10-Q,2025-07-01 to 2025-09-30 (91d),2025-11-04 21:18:08+00:00,1.439300e+10,14393000000,0.0,True
3,JPM,2025,Q4,FY - 9M/YTD candidate,10-K 0001628280-26-008131 | 10-Q 0001628280-25...,10-K | 10-Q,2025-01-01 to 2025-12-31 (364d) | 2025-01-01 t...,57048000000 | 44023000000,2026-02-13 21:20:00+00:00 | 2025-11-04 21:18:0...,None,None,NaT,NaN,13025000000,NaN,True
8,UNH,2025,Q1,direct quarter candidate,10-Q 0000731766-25-000131,10-Q,2025-01-01 to 2025-03-31 (89d),6292000000,2025-05-06 22:44:00+00:00,0000731766-25-000131 / 10-Q,2025-01-01 to 2025-03-31 (89d),2025-05-06 22:44:00+00:00,6.292000e+09,6292000000,NaN,True
9,UNH,2025,Q2,H1/YTD - Q1,10-Q 0000731766-25-000236 | 10-Q 0000731766-25...,10-Q | 10-Q,2025-01-01 to 2025-06-30 (180d) | 2025-01-01 t...,9698000000 | 6292000000,2025-08-11 20:20:29+00:00 | 2025-05-06 22:44:0...,0000731766-25-000236 / 10-Q,2025-04-01 to 2025-06-30 (90d),2025-08-11 20:20:29+00:00,3.406000e+09,3406000000,0.0,True
10,UNH,2025,Q3,9M/YTD - H1/YTD,10-Q 0000731766-25-000305 | 10-Q 0000731766-25...,10-Q | 10-Q,2025-01-01 to 2025-09-30 (272d) | 2025-01-01 t...,12046000000 | 9698000000,2025-10-28 20:09:07+00:00 | 2025-08-11 20:20:2...,0000731766-25-000305 / 10-Q,2025-07-01 to 2025-09-30 (91d),2025-10-28 20:09:07+00:00,2.348000e+09,2348000000,0.0,True
11,UNH,2025,Q4,FY - 9M/YTD candidate,10-K 0000731766-26-000062 | 10-Q 0000731766-25...,10-K | 10-Q,2025-01-01 to 2025-12-31 (364d) | 2025-01-01 t...,12056000000 | 12046000000,2026-03-02 11:06:21+00:00 | 2025-10-28 20:09:0...,None,None,NaT,NaN,10000000,NaN,True
12,WMT,2026,Q1,direct quarter candidate,10-Q 0000104169-25-000090,10-Q,2025-02-01 to 2025-04-30 (88d),4487000000,2025-06-06 20:36:50+00:00,0000104169-25-000090 / 10-Q,2025-02-01 to 2025-04-30 (88d),2025-06-06 20:36:50+00:00,4.487000e+09,4487000000,NaN,True
13,WMT,2026,Q2,H1/YTD - Q1,10-Q 0000104169-25-000137 | 10-Q 0000104169-25...,10-Q | 10-Q,2025-02-01 to 2025-07-31 (180d) | 2025-02-01 t...,11513000000 | 4487000000,2025-08-29 20:13:07+00:00 | 2025-06-06 20:36:5...,0000104169-25-000137 / 10-Q,2025-05-01 to 2025-07-31 (91d),2025-08-29 20:13:07+00:00,7.026000e+09,7026000000,0.0,True


### Adversarial historical stress cases

The recent-year test can pass while the general method still fails governance. This cell deliberately searches the full preserved history for each issuer's `NetIncomeLoss` amendment or repeated economic period. A differing value is an explicit contradiction; an identical value is still a vintage-selection problem because the later filing was not available earlier.

In [16]:
stress_rows = []
for issuer in TEST_ISSUERS:
    history = net_income.loc[net_income['issuer'].eq(issuer)].sort_values('acceptanceDateTime')
    amended = history.loc[history['form'].str.endswith('/A')]
    duplicate_groups = history.loc[history.duplicated(['issuer', 'tag', 'unit', 'start', 'end'], keep=False)]
    if not amended.empty:
        later = amended.iloc[0]
        same_period = history.loc[(history['start'].eq(later['start'])) & (history['end'].eq(later['end'])) & (history['acceptanceDateTime'] <= later['acceptanceDateTime'])]
        earlier = same_period.loc[~same_period['form'].str.endswith('/A')].sort_values('acceptanceDateTime')
        prior = earlier.iloc[0] if not earlier.empty else None
        status = 'AMBIGUOUS' if prior is None else ('FAIL' if prior['value'] != later['value'] else 'PASS WITH CONDITION')
        stress_rows.append({'issuer': issuer, 'case': 'NetIncomeLoss amendment', 'economic_period': f"{later['start'].date()} to {later['end'].date()}", 'earlier accession/value': None if prior is None else f"{prior['accn']} / {prior['value']}", 'later accession/value': f"{later['accn']} / {later['value']}", 'earlier acceptance': None if prior is None else prior['acceptanceDateTime'], 'later acceptance': later['acceptanceDateTime'], 'why adversarial': 'Later amendment cannot rewrite the earlier PIT set', 'status': status})
    elif not duplicate_groups.empty:
        pair = duplicate_groups.iloc[0]
        stress_rows.append({'issuer': issuer, 'case': 'comparative duplicate without NetIncomeLoss amendment', 'economic_period': f"{pair['start'].date()} to {pair['end'].date()}", 'earlier accession/value': 'Inspect preserved competing accessions', 'later accession/value': 'No qualifying amendment fact selected', 'earlier acceptance': None, 'later acceptance': None, 'why adversarial': 'Exact-period duplicates require a vintage rule even when values agree', 'status': 'AMBIGUOUS'})
    else:
        filing_amendments = filings.loc[(filings['issuer'].eq(issuer)) & filings['form'].str.endswith('/A')].shape[0]
        stress_rows.append({'issuer': issuer, 'case': 'no NetIncomeLoss amendment fact in inspected subset', 'economic_period': None, 'earlier accession/value': None, 'later accession/value': None, 'earlier acceptance': None, 'later acceptance': None, 'why adversarial': f'{filing_amendments} amendment filings still require an exception policy', 'status': 'AMBIGUOUS'})

stress_cases = pd.DataFrame(stress_rows)
stress_cases

,issuer,case,economic_period,earlier accession/value,later accession/value,earlier acceptance,later acceptance,why adversarial,status
0,JPM,NetIncomeLoss amendment,2011-01-01 to 2011-03-31,0000950123-11-045948 / 5555000000,0000019617-12-000262 / 5555000000,2011-05-06 01:27:44+00:00,2012-08-09 11:44:21+00:00,Later amendment cannot rewrite the earlier PIT...,PASS WITH CONDITION
1,XOM,NetIncomeLoss amendment,2009-01-01 to 2009-12-31,0001193125-10-042929 / 19280000000,0001193125-11-050134 / 19280000000,2010-02-26 21:17:17+00:00,2011-02-28 22:01:32+00:00,Later amendment cannot rewrite the earlier PIT...,PASS WITH CONDITION
2,UNH,comparative duplicate without NetIncomeLoss am...,2009-04-01 to 2009-06-30,Inspect preserved competing accessions,No qualifying amendment fact selected,NaT,NaT,Exact-period duplicates require a vintage rule...,AMBIGUOUS
3,WMT,NetIncomeLoss amendment,2012-02-01 to 2012-07-31,0000104169-12-000014 / 7758000000,0000104169-13-000039 / 7758000000,2012-09-06 20:25:30+00:00,2013-10-21 21:25:31+00:00,Later amendment cannot rewrite the earlier PIT...,PASS WITH CONDITION


### Falsification interpretation

A `FAIL` means the candidate rule cannot be applied to that issuer-year without an unapproved repair. `PASS WITH CONDITION` means arithmetic and the tested structural checks survived, but duplicate/comparative facts, amendments, calendar precision, or a still-unfrozen vintage rule prevent a general rule from being frozen. No ambiguity is silently repaired.

## 12. Decision

# CONDITIONAL GO

The recent complete-year reconciliations for JPM, XOM, UNH, and WMT pass the tested direct-versus-cumulative arithmetic and structural checks. The adversarial historical cases nevertheless show amendments and/or comparative duplicates. Thus the candidate method is feasible in principle but is **not** a general formula: numerical equality does not resolve filing-vintage selection or prospective amendment treatment.

The Stage 1C Net Income information policy freezes prospective amendments, compatibility checks, the three candidate reconstruction identities, `NA` for unresolved direct/reconstructed differences or non-amendment duplicates, and no imputation. The reusable implementation below applies only this quarterly/TTM Net Income policy.

No Earnings Yield, Market Cap, Value score, `X_char`, `w_raw`, `F_risk`, backtest, or performance-based selection was implemented.

## 13. Reusable implementation consistency check

The transparent AAPL trace below calls the reusable PIT quarterly/TTM implementation and asserts its manually recorded values. It is an implementation-consistency check, not a second production implementation.

## 14. Frozen SEC acceptance-to-XNYS eligibility

The reusable utility applies the frozen rule: a filing is eligible at the opening of the first XNYS regular session whose opening is **strictly after** its timezone-aware SEC `acceptanceDateTime`. The acceptance timestamp, session label, and portfolio decision time remain different fields. Scheduled early-close sessions are eligible. This is a conservative research eligibility convention, not evidence of the exact SEC.gov dissemination instant.

In [ ]:
import sys

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from xnys_pit import eligibility_from_acceptance

eligibility_examples = pd.DataFrame([
    ('before regular open', '2025-06-03T13:00:00Z', '2025-06-03', 'The same session opens later.'),
    ('during regular session', '2025-06-03T15:00:00Z', '2025-06-04', 'That session is already in progress.'),
    ('after regular close', '2025-06-03T21:30:00Z', '2025-06-04', 'The next session is required.'),
    ('Friday after close', '2025-06-06T21:30:00Z', '2025-06-09', 'Weekend sessions do not exist.'),
    ('weekend', '2025-06-07T15:00:00Z', '2025-06-09', 'The next eligible session is Monday.'),
    ('before an early-close session', '2025-07-03T12:00:00Z', '2025-07-03', 'The scheduled early-close session remains eligible.'),
    ('after early close before holiday', '2025-07-03T18:00:00Z', '2025-07-07', 'July 4 is closed; the next session is Monday.'),
], columns=['case', 'acceptanceDateTime', 'expected_session', 'why'])

eligibility_examples['result'] = eligibility_examples['acceptanceDateTime'].map(eligibility_from_acceptance)
eligibility_examples['eligible_XNYS_session'] = eligibility_examples['result'].map(lambda x: str(x.eligibility_session.date()))
eligibility_examples['portfolio_decision_time_utc'] = eligibility_examples['result'].map(lambda x: x.portfolio_decision_time.isoformat())
assert (eligibility_examples['eligible_XNYS_session'] == eligibility_examples['expected_session']).all()
eligibility_examples[['case', 'acceptanceDateTime', 'eligible_XNYS_session', 'portfolio_decision_time_utc', 'why']]

## 15. AAPL end-to-end PIT TTM trace

The three decision times bracket AAPL's FY2025 10-K acceptance (`2025-10-31T10:01:26Z`). At the regular-session opening on 2025-10-30 the 10-K is not yet eligible. It becomes eligible at the 2025-10-31 XNYS opening; the later 2025-11-03 decision has no new fundamental event. Each displayed record retains the selected economic quarter, direct/reconstructed status, accession(s), acceptance timestamp(s), and calculated PIT eligibility.

In [ ]:
from pit_net_income import build_quarters_as_of, ttm_net_income_as_of

AAPL_DECISION_TIMES = [
    '2025-10-30T13:30:00Z',  # immediately before FY2025 10-K eligibility
    '2025-10-31T13:30:00Z',  # first eligible XNYS opening
    '2025-11-03T14:30:00Z',  # later session, before the next fundamental event
]
# These manually recorded sums are visible consequences of the raw AAPL trace above.
MANUAL_AAPL_TTM = {
    '2025-10-30T13:30:00Z': 99_280_000_000,
    '2025-10-31T13:30:00Z': 112_010_000_000,
    '2025-11-03T14:30:00Z': 112_010_000_000,
}

aapl_trace_rows = []
for decision_time in AAPL_DECISION_TIMES:
    all_quarters = build_quarters_as_of(net_income, 'AAPL', decision_time)
    result = ttm_net_income_as_of(net_income, 'AAPL', decision_time)
    assert result.status == 'VALID'
    assert result.value == MANUAL_AAPL_TTM[decision_time]
    for quarter in result.quarters.itertuples(index=False):
        aapl_trace_rows.append({
            'portfolio decision time': result.decision_time,
            'available quarter records': len(all_quarters),
            'economic quarter': f'{quarter.economic_quarter_start.date()} to {quarter.economic_quarter_end.date()}',
            'method': quarter.method,
            'quarter value': quarter.value,
            'source accession(s)': ' | '.join(quarter.source_accessions),
            'source acceptance timestamp(s)': ' | '.join(x.isoformat() for x in quarter.source_acceptance_datetimes),
            'PIT eligibility opening(s)': ' | '.join(x.isoformat() for x in quarter.pit_eligibility),
            'NI_TTM_PIT': result.value,
        })

aapl_pit_ttm_trace = pd.DataFrame(aapl_trace_rows)
aapl_pit_ttm_trace

AAPL's PIT TTM changes from $99.280bn to $112.010bn only when the FY2025 filing crosses its eligibility opening. It is unchanged on the subsequent market session because no new eligible SEC Net Income information entered the information set. The first date cannot use the future FY2025 10-K.

## 16. Cross-issuer implementation validation

Each date is the first eligible session opening for the selected most-recent complete-year 10-K in the preserved audit sample. This validates the deterministic implementation only; it is not a new factor study.

In [ ]:
CROSS_ISSUER_DECISION_TIMES = {
    'JPM': '2026-02-17T14:30:00Z',
    'XOM': '2026-02-19T14:30:00Z',
    'UNH': '2026-03-02T14:30:00Z',
    'WMT': '2026-03-16T13:30:00Z',
}

cross_rows = []
for issuer, decision_time in CROSS_ISSUER_DECISION_TIMES.items():
    result = ttm_net_income_as_of(net_income, issuer, decision_time)
    cross_rows.append({
        'issuer': issuer,
        'representative decision date': result.decision_time,
        'four selected economic quarters': ' | '.join(x.strftime('%Y-%m-%d') for x in result.quarters['economic_quarter_end']),
        'NI_TTM_PIT': result.value,
        'NA / ambiguity encountered': None if result.status == 'VALID' else result.status,
        'validation status': result.status,
    })

cross_issuer_ttm_validation = pd.DataFrame(cross_rows)
assert cross_issuer_ttm_validation['validation status'].eq('VALID').all()
cross_issuer_ttm_validation

### NA and ambiguity result

No `NA` was encountered in the five representative current-vintage TTM validations. This is not a claim that all historical issuer-quarters are usable: the earlier AAPL/JPM/XOM/UNH/WMT stress cases remain evidence that a missing compatible component, unresolved duplicate, or direct/reconstructed mismatch must produce `NA`, and then fewer than four valid consecutive quarters produces no TTM.

## 17. Price/share-basis compatibility audit — not Market Cap

This section does **not** construct Market Cap. It audits whether the preserved Alpha Vantage fields share the same corporate-action basis as the frozen latest PIT-eligible SEC `dei:EntityCommonStockSharesOutstanding` state. Diagnostic price-times-shares products are displayed only to expose basis mismatches. At a portfolio open on `t`, each price date shown is an already-completed session and the shares record is selected only when its own SEC filing is PIT-eligible by that following decision opening.

In [ ]:
from xnys_pit import XNYS

AV_RAW_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'stage_0'
av_manifest = json.loads((AV_RAW_ROOT / 'manifest.json').read_text(encoding='utf-8'))

def alpha_vantage_response(symbol, endpoint):
    candidates = [x for x in av_manifest if x['symbol'] == symbol and x['endpoint'] == endpoint and x['request_status'] == 'HTTP_200']
    assert candidates, f'missing preserved {endpoint} response for {symbol}'
    path = PROJECT_ROOT / candidates[-1]['local_raw_file']
    return json.loads(path.read_text(encoding='utf-8'))

def daily_adjusted_prices(symbol):
    payload = alpha_vantage_response(symbol, 'TIME_SERIES_DAILY_ADJUSTED')
    prices = pd.DataFrame.from_dict(payload['Time Series (Daily)'], orient='index')
    prices.index = pd.to_datetime(prices.index)
    prices.index.name = 'price_date'
    prices = prices.rename(columns={
        '1. open': 'open', '2. high': 'high', '3. low': 'low', '4. close': 'close',
        '5. adjusted close': 'adjusted_close', '6. volume': 'volume',
        '7. dividend amount': 'dividend_amount', '8. split coefficient': 'split_coefficient',
    }).apply(pd.to_numeric)
    return prices.sort_index()

def sec_reported_shares():
    rows = []
    for issuer, payload in facts_payloads.items():
        observations = payload.get('facts', {}).get('dei', {}).get('EntityCommonStockSharesOutstanding', {}).get('units', {}).get('shares', [])
        for fact in observations:
            if fact.get('form') in FORMS:
                rows.append({
                    'issuer': issuer, 'cik': str(payload['cik']).zfill(10),
                    'shares_observation_date': fact.get('end'), 'reported_shares': fact.get('val'),
                    'form': fact.get('form'), 'filed': fact.get('filed'), 'accn': fact.get('accn'),
                })
    shares = pd.DataFrame(rows).merge(
        filings.rename(columns={'accessionNumber': 'accn', 'form': 'filing_form'}),
        on=['issuer', 'accn'], how='left', validate='many_to_one'
    )
    shares['shares_observation_date'] = pd.to_datetime(shares['shares_observation_date'])
    shares['acceptanceDateTime'] = pd.to_datetime(shares['acceptanceDateTime'], utc=True, errors='coerce')
    shares['pit_eligibility'] = shares['acceptanceDateTime'].map(
        lambda accepted: eligibility_from_acceptance(accepted).portfolio_decision_time if pd.notna(accepted) else pd.NaT
    )
    return shares

SEC_REPORTED_SHARES = sec_reported_shares()

def following_decision_open(price_date):
    session = XNYS.date_to_session(pd.Timestamp(price_date), direction='none')
    return XNYS.session_open(XNYS.next_session(session))

def latest_pit_reported_shares(issuer, decision_open):
    eligible = SEC_REPORTED_SHARES.loc[
        SEC_REPORTED_SHARES['issuer'].eq(issuer)
        & SEC_REPORTED_SHARES['pit_eligibility'].notna()
        & SEC_REPORTED_SHARES['pit_eligibility'].le(decision_open)
    ].sort_values(['pit_eligibility', 'acceptanceDateTime', 'accn'])
    assert not eligible.empty, f'no PIT-eligible SEC shares for {issuer} at {decision_open}'
    # This explicitly applies the frozen reported-shares-as-of convention; it is not an imputation.
    return eligible.iloc[-1]


### AAPL: 4-for-1 split on 2020-08-31

The raw Alpha Vantage observations identify the event through `split_coefficient = 4.0` on 2020-08-31. The table includes two pre/post split dates and a later date after a new SEC shares observation becomes eligible. The final two columns are diagnostic products, not constructed market-capitalization data.

In [ ]:
aapl_prices = daily_adjusted_prices('AAPL')
AAPL_AUDIT_DATES = pd.to_datetime(['2020-08-28', '2020-08-31', '2020-09-01', '2020-10-29'])
aapl_split_window = aapl_prices.loc['2020-08-24':'2020-09-04'].copy()

aapl_rows = []
for price_date in AAPL_AUDIT_DATES:
    decision_open = following_decision_open(price_date)
    shares = latest_pit_reported_shares('AAPL', decision_open)
    price = aapl_prices.loc[price_date]
    aapl_rows.append({
        'price date (completed session)': price_date,
        'following portfolio decision open': decision_open,
        'open': price['open'], 'high': price['high'], 'low': price['low'],
        'close': price['close'], 'adjusted close': price['adjusted_close'],
        'split coefficient': price['split_coefficient'], 'volume': price['volume'],
        'SEC shares observation date': shares['shares_observation_date'],
        'SEC shares accession': shares['accn'],
        'SEC shares acceptance': shares['acceptanceDateTime'],
        'SEC shares PIT eligibility': shares['pit_eligibility'],
        'Shares_PIT (reported state)': shares['reported_shares'],
        'raw close × Shares_PIT (diagnostic)': price['close'] * shares['reported_shares'],
        'adjusted close × Shares_PIT (diagnostic)': price['adjusted_close'] * shares['reported_shares'],
    })

aapl_price_share_audit = pd.DataFrame(aapl_rows)
assert aapl_prices.loc[pd.Timestamp('2020-08-31'), 'split_coefficient'] == 4.0
assert aapl_prices.loc[pd.Timestamp('2020-08-28'), 'close'] > 3 * aapl_prices.loc[pd.Timestamp('2020-08-31'), 'close']
assert aapl_prices.loc[pd.Timestamp('2020-08-28'), 'adjusted_close'] < aapl_prices.loc[pd.Timestamp('2020-08-28'), 'close'] / 3
display(aapl_split_window[['open', 'high', 'low', 'close', 'adjusted_close', 'split_coefficient', 'volume']])
aapl_price_share_audit

**Observed basis behavior.** Before the split, raw close remains on the approximately 4.3bn-share basis reported by the eligible July filing; adjusted close is already roughly one-quarter of raw close, revealing retrospective split adjustment (and it also reflects dividend adjustment). Immediately after the split, the frozen latest reported SEC share state is still pre-split, so neither diagnostic product is economically reliable until the later 10-K shares observation (about 17.0bn shares) becomes eligible. After that observation, raw close and reported shares share the post-split basis. Adjusted close remains dividend/split adjusted and is not the historical per-share transaction-price basis.

### Cross-check: WMT 3-for-1 split on 2024-02-26

WMT supplies a second preserved split case. It tests the same question without expanding into a broad corporate-actions study.

In [ ]:
wmt_prices = daily_adjusted_prices('WMT')
WMT_AUDIT_DATES = pd.to_datetime(['2024-02-23', '2024-02-26', '2024-02-27', '2024-03-18'])
wmt_rows = []
for price_date in WMT_AUDIT_DATES:
    decision_open = following_decision_open(price_date)
    shares = latest_pit_reported_shares('WMT', decision_open)
    price = wmt_prices.loc[price_date]
    wmt_rows.append({
        'price date': price_date, 'following decision open': decision_open,
        'close': price['close'], 'adjusted close': price['adjusted_close'],
        'split coefficient': price['split_coefficient'],
        'SEC shares observation date': shares['shares_observation_date'],
        'SEC shares accession': shares['accn'], 'Shares_PIT': shares['reported_shares'],
        'raw close × Shares_PIT (diagnostic)': price['close'] * shares['reported_shares'],
        'adjusted close × Shares_PIT (diagnostic)': price['adjusted_close'] * shares['reported_shares'],
    })

wmt_price_share_audit = pd.DataFrame(wmt_rows)
assert wmt_prices.loc[pd.Timestamp('2024-02-26'), 'split_coefficient'] == 3.0
wmt_price_share_audit

## 18. Decision: CONDITIONAL GO for raw close only

Candidate future formula, **not implemented here**:

`MarketCap_(i,t) = raw_close_(i,t-1) × Shares_PIT_(i,t)`

`raw_close` is the only audited field that retains the observed pre/post split price basis. `adjusted_close` is rejected for this object because the preserved history retrospectively incorporates split information into pre-split observations and additionally reflects dividend adjustment. The GO is conditional: a construction must return `NA`, rather than silently combine values, whenever a split/coefficient event has changed the price basis after the latest PIT-eligible reported SEC shares observation and before the price date. The frozen reusable implementation below applies that gate.

## 19. Frozen PIT Market Cap and raw Earnings Yield

At the XNYS open on `t`, the reusable implementation selects only the completed raw close from the previous eligible session (`t-1`) and the latest PIT-eligible SEC reported-share state. It returns an explicit status rather than silently combining incompatible bases. Raw Value is now defined as `EarningsYield = NI_TTM_PIT / MarketCap_PIT`; it remains a raw issuer-level input, not a rank, score, or `X_char` column.

In [ ]:
from pit_value import earnings_yield_as_of, market_cap_as_of

AAPL_MARKET_CAP_DECISIONS = [
    ('before 4-for-1 split', '2020-08-31T13:30:00Z'),
    ('split-basis mismatch', '2020-09-01T13:30:00Z'),
    ('post-split shares eligible', '2020-10-30T13:30:00Z'),
    ('recent valid Value example', '2025-11-03T14:30:00Z'),
]

aapl_value_rows = []
for case, decision_time in AAPL_MARKET_CAP_DECISIONS:
    market_cap = market_cap_as_of(aapl_prices, SEC_REPORTED_SHARES, 'AAPL', decision_time)
    value = earnings_yield_as_of(net_income, aapl_prices, SEC_REPORTED_SHARES, 'AAPL', decision_time)
    aapl_value_rows.append({
        'case': case, 'decision time': market_cap.decision_time,
        'raw price date': market_cap.raw_price_date, 'raw close': market_cap.raw_close,
        'Shares_PIT': market_cap.shares_value, 'shares accession': market_cap.shares_source_accession,
        'shares reference date': market_cap.shares_reference_date,
        'shares PIT eligibility': market_cap.shares_pit_eligibility,
        'split event': market_cap.split_event_date, 'MarketCap_PIT': market_cap.market_cap,
        'NI_TTM_PIT': value.net_income.value, 'EarningsYield': value.earnings_yield,
        'status': value.status, 'reason': value.reason,
    })

aapl_value_trace = pd.DataFrame(aapl_value_rows)
assert aapl_value_trace.loc[0, 'MarketCap_PIT'] == 2_134_524_761_820.0
assert aapl_value_trace.loc[1, 'status'] == 'SPLIT_BASIS_MISMATCH'
assert pd.isna(aapl_value_trace.loc[1, 'MarketCap_PIT']) and pd.isna(aapl_value_trace.loc[1, 'EarningsYield'])
assert aapl_value_trace.loc[2, 'MarketCap_PIT'] == 1_960_647_806_640.0
aapl_value_trace

### Recent AAPL visible calculation and reusable-consistency assertion

The recent example shows the numerator, denominator, and their quotient explicitly. The assertion compares that transparent arithmetic with the reusable result; it does not define a second implementation.

In [ ]:
recent_aapl = earnings_yield_as_of(net_income, aapl_prices, SEC_REPORTED_SHARES, 'AAPL', '2025-11-03T14:30:00Z')
assert recent_aapl.status == 'VALID'
visible_market_cap = recent_aapl.market_cap.raw_close * recent_aapl.market_cap.shares_value
visible_earnings_yield = recent_aapl.net_income.value / visible_market_cap
assert visible_market_cap == recent_aapl.market_cap.market_cap
assert visible_earnings_yield == recent_aapl.earnings_yield
pd.DataFrame([{
    'decision time': recent_aapl.decision_time,
    'NI_TTM_PIT': recent_aapl.net_income.value,
    'raw_close_(t-1)': recent_aapl.market_cap.raw_close,
    'Shares_PIT': recent_aapl.market_cap.shares_value,
    'MarketCap_PIT': visible_market_cap,
    'EarningsYield': visible_earnings_yield,
    'status': recent_aapl.status,
}])

## 20. Cross-issuer raw Value implementation validation

One representative recent decision time is checked per issuer. These rows are neither compared nor ranked; they validate only that the frozen PIT numerator, denominator, split gate, and provenance path operate together.

In [ ]:
VALUE_VALIDATION_DATES = {
    'AAPL': '2025-11-03T14:30:00Z',
    'JPM': '2026-02-17T14:30:00Z',
    'XOM': '2026-02-19T14:30:00Z',
    'UNH': '2026-03-02T14:30:00Z',
    'WMT': '2026-03-16T13:30:00Z',
}

value_validation_rows = []
for issuer, decision_time in VALUE_VALIDATION_DATES.items():
    result = earnings_yield_as_of(net_income, daily_adjusted_prices(issuer), SEC_REPORTED_SHARES, issuer, decision_time)
    value_validation_rows.append({
        'issuer': issuer, 'decision date': result.decision_time,
        'NI_TTM_PIT': result.net_income.value,
        'MarketCap_PIT': result.market_cap.market_cap,
        'EarningsYield': result.earnings_yield,
        'status': result.status,
    })

raw_value_validation = pd.DataFrame(value_validation_rows)
assert raw_value_validation['status'].eq('VALID').all()
raw_value_validation